In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/reddit_data/raw_yearly"

In [ ]:
import os
import re
import json
from datetime import datetime
from collections import defaultdict
import pandas as pd

In [ ]:
# Picking one subreddit folder to test first
SUBREDDIT = "MachineLearning"
subreddit_dir = os.path.join(DATA_ROOT, SUBREDDIT)

print("Folder exists:", os.path.exists(subreddit_dir))
print("Sample files:", os.listdir(subreddit_dir)[:5])

In [ ]:
token_pattern = re.compile(r"[a-zA-Z][a-zA-Z0-9_'-]*")

def stream_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue

def get_month(created_utc):
    try:
        dt = datetime.utcfromtimestamp(int(created_utc))
        return dt.strftime("%Y-%m")
    except Exception:
        return None

def tokenize(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    return token_pattern.findall(text)

In [ ]:
word_counts = defaultdict(int)

files = sorted(os.listdir(subreddit_dir))[:2]   # test on first 2 files only
print("Testing files:", files)

for fname in files:
    fpath = os.path.join(subreddit_dir, fname)
    print("Processing:", fpath)

    for comment in stream_jsonl(fpath):
        body = comment.get("body", "")
        created_utc = comment.get("created_utc")

        month = get_month(created_utc)
        if month is None:
            continue

        tokens = tokenize(body)
        for token in tokens:
            word_counts[(token, month)] += 1

print("Unique word-month pairs:", len(word_counts))

In [ ]:
rows = [
    {"word": word, "month": month, "count": count}
    for (word, month), count in word_counts.items()
]

df = pd.DataFrame(rows)
df = df.sort_values(["word", "month"]).reset_index(drop=True)

print(df.head(20))
print(df[df["word"] == "model"].tail(20))

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/reddit_data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

out_path = os.path.join(OUTPUT_DIR, f"{SUBREDDIT}_word_counts_test.parquet")
df.to_parquet(out_path, index=False)

print("Saved to:", out_path)

In [ ]:
word_counts = defaultdict(int)

files = sorted(os.listdir(subreddit_dir))
print("Total files:", len(files))

for i, fname in enumerate(files, start=1):
    fpath = os.path.join(subreddit_dir, fname)
    print(f"[{i}/{len(files)}] Processing:", fname)

    for comment in stream_jsonl(fpath):
        body = comment.get("body", "")
        created_utc = comment.get("created_utc")

        month = get_month(created_utc)
        if month is None:
            continue

        tokens = tokenize(body)
        for token in tokens:
            word_counts[(token, month)] += 1

rows = [
    {"word": word, "month": month, "count": count}
    for (word, month), count in word_counts.items()
]

df = pd.DataFrame(rows).sort_values(["word", "month"]).reset_index(drop=True)

out_path = os.path.join(OUTPUT_DIR, f"{SUBREDDIT}_word_counts.parquet")
df.to_parquet(out_path, index=False)

print("Saved full file to:", out_path)
print("Rows:", len(df))

In [ ]:
for w in ["model", "agent", "token", "prompt"]:
    print("\nWORD:", w)
    print(df[df["word"] == w].tail(12))

build appearance + breakout detection

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def get_word_timeline(df, word):
    out = df[df["word"] == word].copy()
    out = out.sort_values("month").reset_index(drop=True)
    return out

In [ ]:
def find_appearance_month(df, word, min_count=20, sustain_months=2):
    timeline = get_word_timeline(df, word)

    if timeline.empty:
        return None, None

    counts = timeline["count"].tolist()
    months = timeline["month"].tolist()

    for i in range(len(counts) - sustain_months + 1):
        window = counts[i:i+sustain_months]
        if all(c >= min_count for c in window):
            return months[i], timeline.iloc[i:i+sustain_months]

    return None, None

In [ ]:
def find_breakout_month(df, word, appearance_month, min_count=30, growth_ratio=3.0, lookback=3):

    timeline = get_word_timeline(df, word)

    if timeline.empty:
        return None, None

    counts = timeline["count"].tolist()
    months = timeline["month"].tolist()

    for i in range(lookback, len(counts)):

        # ignore months before appearance
        if months[i] < appearance_month:
            continue

        prev_avg = np.mean(counts[i-lookback:i])
        curr = counts[i]

        if prev_avg > 0 and curr >= min_count and (curr / prev_avg) >= growth_ratio:

            return months[i], {
                "month": months[i],
                "count": curr,
                "prev_avg": prev_avg,
                "growth_ratio": curr / prev_avg
            }

    return None, None

appearance_month, _ = find_appearance_month(df, w)

breakout_month, breakout_info = find_breakout_month(
    df,
    w,
    appearance_month
)

In [ ]:
test_words = ["model", "agent", "token", "prompt"]

for w in test_words:

    appearance_month, appearance_window = find_appearance_month(
        df, w, min_count=20, sustain_months=2
    )

    breakout_month, breakout_info = find_breakout_month(
        df,
        w,
        appearance_month,
        min_count=30,
        growth_ratio=3.0,
        lookback=3
    )

    print(f"\nWORD: {w}")
    print("Appearance:", appearance_month)
    print("Breakout:", breakout_month)
    print("Breakout info:", breakout_info)

In [ ]:
timeline_model = get_word_timeline(df, "model")
print(timeline_model.head(30))
print(timeline_model.tail(30))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_word_timeline(df, word):

    timeline = get_word_timeline(df, word)

    if timeline.empty:
        print("No data")
        return

    months = timeline["month"]
    counts = timeline["count"]

    plt.figure(figsize=(14,5))
    plt.plot(months, counts)

    # show fewer ticks
    step = max(1, len(months)//10)
    plt.xticks(months[::step], rotation=45)

    plt.title(f"Monthly counts for '{word}'")
    plt.xlabel("Month")
    plt.ylabel("Count")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_word_timeline(df, "model")
plot_word_timeline(df, "agent")
plot_word_timeline(df, "prompt")

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/reddit_data/processed"

CONTEXT_DIR = os.path.join(PROJECT_ROOT, "contexts")
EMBED_DIR = os.path.join(PROJECT_ROOT, "embeddings")
CLUSTER_DIR = os.path.join(PROJECT_ROOT, "clusters")

os.makedirs(CONTEXT_DIR, exist_ok=True)
os.makedirs(EMBED_DIR, exist_ok=True)
os.makedirs(CLUSTER_DIR, exist_ok=True)

print("Directories ready")

In [ ]:
import os
import json
import re
import numpy as np
import pandas as pd
import hdbscan

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [ ]:
# ---------- Basic config ----------
SUBREDDIT = "MachineLearning"
subreddit_dir = os.path.join(DATA_ROOT, SUBREDDIT)

target_words = [
    "prompt",
    "token",
    "alignment",
    "hallucination",
    "copilot",
    "agent",
    "model"
]

OLD_END = "2019-12"
NEW_START = "2023-01"

PER_MONTH_LIMIT = 200
WINDOW_SIZE = 40
MIN_CHARS = 40

HDBSCAN_MIN_CLUSTER_SIZE = 20
HDBSCAN_MIN_SAMPLES = 5

In [ ]:
def stream_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                yield json.loads(line)
            except:
                continue

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def get_month(created_utc):
    try:
        return pd.to_datetime(created_utc, unit="s").strftime("%Y-%m")
    except:
        return None

def extract_local_window(text, word, window_size=40):
    tokens = text.split()
    word = word.lower()

    positions = [i for i, tok in enumerate(tokens) if tok == word]
    if not positions:
        return []

    windows = []
    for i in positions:
        start = max(0, i - window_size)
        end = min(len(tokens), i + window_size + 1)
        windows.append(" ".join(tokens[start:end]))
    return windows

def extract_local_contexts_balanced(
    word,
    subreddit_dir,
    subreddit_name,
    per_month_limit=200,
    window_size=40,
    min_chars=40
):
    rows = []
    month_counts = {}
    files = sorted(os.listdir(subreddit_dir))

    for i, fname in enumerate(files, start=1):
        if not fname.endswith(".jsonl"):
            continue

        fpath = os.path.join(subreddit_dir, fname)
        print(f"[{word}] [{i}/{len(files)}] scanning {fname}")

        for comment in stream_jsonl(fpath):
            body = clean_text(comment.get("body", ""))
            if len(body) < min_chars:
                continue

            if f" {word} " not in f" {body} ":
                continue

            month = get_month(comment.get("created_utc"))
            if month is None:
                continue

            if month not in month_counts:
                month_counts[month] = 0

            if month_counts[month] >= per_month_limit:
                continue

            windows = extract_local_window(body, word, window_size=window_size)
            if not windows:
                continue

            for w in windows:
                if month_counts[month] >= per_month_limit:
                    break
                if len(w) < min_chars:
                    continue

                rows.append({
                    "word": word,
                    "month": month,
                    "subreddit": subreddit_name,
                    "text": w
                })
                month_counts[month] += 1

    df_out = pd.DataFrame(rows)
    if not df_out.empty:
        df_out = df_out.drop_duplicates(subset=["text"]).reset_index(drop=True)
    return df_out

def compute_drift_score(ctx_df, emb, old_end="2019-12", new_start="2023-01"):
    old_mask = ctx_df["month"] <= old_end
    new_mask = ctx_df["month"] >= new_start

    old_count = int(old_mask.sum())
    new_count = int(new_mask.sum())

    if old_count == 0 or new_count == 0:
        return None, old_count, new_count

    old_centroid = np.mean(emb[old_mask.values], axis=0)
    new_centroid = np.mean(emb[new_mask.values], axis=0)

    drift_score = 1 - cosine_similarity([old_centroid], [new_centroid])[0][0]
    return float(drift_score), old_count, new_count

def assign_period(month):
    if month <= OLD_END:
        return "old"
    elif month >= NEW_START:
        return "new"
    return "middle"

In [ ]:
embed_model = SentenceTransformer("all-mpnet-base-v2")

In [ ]:
contexts_by_word = {}
embeddings_by_word = {}
drift_scores_by_word = {}
clustered_dfs_by_word = {}
cluster_period_tables_by_word = {}
cluster_period_props_by_word = {}
summary_rows = []

In [ ]:
for word in target_words:
    print(f"\n{'='*80}")
    print(f"PROCESSING WORD: {word}")
    print(f"{'='*80}")

    # 1) Extract contexts
    ctx = extract_local_contexts_balanced(
        word=word,
        subreddit_dir=subreddit_dir,
        subreddit_name=SUBREDDIT,
        per_month_limit=PER_MONTH_LIMIT,
        window_size=WINDOW_SIZE,
        min_chars=MIN_CHARS
    )

    contexts_by_word[word] = ctx

    if ctx.empty:
        print(f"[{word}] No contexts found.")
        summary_rows.append({
            "word": word,
            "n_contexts": 0,
            "min_month": None,
            "max_month": None,
            "old_count": 0,
            "new_count": 0,
            "drift_score": None,
            "n_clusters": 0,
            "noise_points": 0
        })
        continue

    print(f"[{word}] contexts shape: {ctx.shape}")
    print(f"[{word}] month range: {ctx['month'].min()} -> {ctx['month'].max()}")

    # SAVE CONTEXTS IMMEDIATELY
    context_path = os.path.join(CONTEXT_DIR, f"{word}_contexts.csv")
    ctx.to_csv(context_path, index=False)
    print(f"[{word}] Saved contexts: {context_path}")

    # 2) Embed
    texts = ctx["text"].tolist()
    emb = embed_model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    embeddings_by_word[word] = emb
    print(f"[{word}] embeddings shape: {emb.shape}")

    # SAVE EMBEDDINGS IMMEDIATELY
    embed_path = os.path.join(EMBED_DIR, f"{word}_embeddings.npy")
    np.save(embed_path, emb)
    print(f"[{word}] Saved embeddings: {embed_path}")

    # 3) Drift
    drift_score, old_count, new_count = compute_drift_score(
        ctx, emb, old_end=OLD_END, new_start=NEW_START
    )
    drift_scores_by_word[word] = drift_score
    print(f"[{word}] old_count={old_count}, new_count={new_count}, drift={drift_score}")

    # 4) HDBSCAN
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples=HDBSCAN_MIN_SAMPLES,
        metric="euclidean"
    )
    labels = clusterer.fit_predict(emb)

    ctx_clustered = ctx.copy()
    ctx_clustered["cluster"] = labels
    ctx_clustered["period"] = ctx_clustered["month"].apply(assign_period)

    clustered_dfs_by_word[word] = ctx_clustered

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_points = int(np.sum(labels == -1))

    print(f"[{word}] n_clusters={n_clusters}, noise_points={noise_points}")

    # SAVE CLUSTERS IMMEDIATELY
    cluster_path = os.path.join(CLUSTER_DIR, f"{word}_clusters.csv")
    ctx_clustered.to_csv(cluster_path, index=False)
    print(f"[{word}] Saved clusters: {cluster_path}")

    # 5) Cluster tables
    cluster_period_table = pd.crosstab(ctx_clustered["cluster"], ctx_clustered["period"])
    cluster_period_prop = pd.crosstab(
        ctx_clustered["cluster"],
        ctx_clustered["period"],
        normalize="columns"
    )

    cluster_period_tables_by_word[word] = cluster_period_table
    cluster_period_props_by_word[word] = cluster_period_prop

    # OPTIONAL: save cluster tables too
    cluster_table_path = os.path.join(CLUSTER_DIR, f"{word}_cluster_period_counts.csv")
    cluster_prop_path = os.path.join(CLUSTER_DIR, f"{word}_cluster_period_props.csv")

    cluster_period_table.to_csv(cluster_table_path)
    cluster_period_prop.to_csv(cluster_prop_path)

    print(f"[{word}] Saved cluster period counts: {cluster_table_path}")
    print(f"[{word}] Saved cluster period props: {cluster_prop_path}")

    # 6) Summary row
    summary_rows.append({
        "word": word,
        "n_contexts": len(ctx),
        "min_month": ctx["month"].min(),
        "max_month": ctx["month"].max(),
        "old_count": old_count,
        "new_count": new_count,
        "drift_score": drift_score,
        "n_clusters": n_clusters,
        "noise_points": noise_points
    })

    # OPTIONAL: save summary after each word too
    partial_summary_df = pd.DataFrame(summary_rows)
    partial_summary_path = os.path.join(PROJECT_ROOT, "partial_summary.csv")
    partial_summary_df.to_csv(partial_summary_path, index=False)
    print(f"[{word}] Updated partial summary: {partial_summary_path}")

In [ ]:
def inspect_word_clusters(word, n_samples=8):
    if word not in clustered_dfs_by_word:
        print(f"No clustered data found for: {word}")
        return

    dfw = clustered_dfs_by_word[word]

    print(f"\nWORD: {word}")
    print("Drift score:", drift_scores_by_word.get(word))
    print("\nCluster counts:")
    print(dfw["cluster"].value_counts().sort_index())

    print("\nCluster by period (raw counts):")
    print(cluster_period_tables_by_word[word])

    print("\nCluster by period (proportions):")
    print(cluster_period_props_by_word[word])

    for c in sorted(dfw["cluster"].unique()):
        if c == -1:
            continue
        print(f"\n===== {word.upper()} CLUSTER {c} =====")
        sample = dfw[dfw["cluster"] == c][["month", "text"]].sample(
            min(n_samples, (dfw["cluster"] == c).sum()),
            random_state=42
        )
        display(sample)

In [ ]:
inspect_word_clusters("prompt")
inspect_word_clusters("agent")
inspect_word_clusters("model")

In [ ]:
def show_old_new_samples(word, n=5):
    if word not in contexts_by_word:
        print(f"No contexts found for: {word}")
        return

    ctx = contexts_by_word[word]
    old_mask = ctx["month"] <= OLD_END
    new_mask = ctx["month"] >= NEW_START

    print(f"\n===== {word.upper()} OLD SAMPLES =====")
    if int(old_mask.sum()) > 0:
        display(ctx[old_mask][["month", "text"]].sample(min(n, int(old_mask.sum())), random_state=42))
    else:
        print("No old samples")

    print(f"\n===== {word.upper()} NEW SAMPLES =====")
    if int(new_mask.sum()) > 0:
        display(ctx[new_mask][["month", "text"]].sample(min(n, int(new_mask.sum())), random_state=42))
    else:
        print("No new samples")

In [ ]:
show_old_new_samples("hallucination")
show_old_new_samples("alignment")

Get appearance and breakout for prompt

In [ ]:
word = "prompt"

appearance_month, appearance_window = find_appearance_month(
    df, word, min_count=20, sustain_months=2
)

breakout_month, breakout_info = find_breakout_month(
    df,
    word,
    appearance_month,
    min_count=30,
    growth_ratio=3.0,
    lookback=3
)

print("Appearance:", appearance_month)
print("Breakout:", breakout_month)
print("Breakout info:", breakout_info)

In [ ]:
import matplotlib.pyplot as plt

w = "prompt"
series = df[df["word"] == w].set_index("month")["count"]

plt.figure(figsize=(10,4))
series.plot()
plt.axvline("2020-07", linestyle="--")
plt.title("Usage of 'prompt' over time (MachineLearning)")
plt.show()

In [ ]:
def analyze_word_live(
    word,
    df_counts,
    subreddit_dir,
    subreddit_name,
    per_month_limit=200,
    window_size=40,
    min_chars=40,
    old_end="2019-12",
    new_start="2023-01",
    min_cluster_size=20,
    min_samples=5
):
    word = word.lower().strip()

    # 1) appearance / breakout
    appearance_month, _ = find_appearance_month(
        df_counts, word, min_count=20, sustain_months=2
    )

    breakout_month, breakout_info = find_breakout_month(
        df_counts, word, appearance_month,
        min_count=30, growth_ratio=3.0, lookback=3
    )

    # 2) contexts
    ctx = extract_local_contexts_balanced(
        word=word,
        subreddit_dir=subreddit_dir,
        subreddit_name=subreddit_name,
        per_month_limit=per_month_limit,
        window_size=window_size,
        min_chars=min_chars
    )

    if ctx.empty:
        return {
            "word": word,
            "status": "no_contexts_found",
            "appearance": appearance_month,
            "breakout": breakout_month,
            "breakout_info": breakout_info
        }

    # 3) embeddings
    texts = ctx["text"].tolist()
    emb = embed_model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    # 4) drift
    drift_score, old_count, new_count = compute_drift_score(
        ctx, emb, old_end=old_end, new_start=new_start
    )

    # 5) evidence check
    if old_count < 20 or new_count < 20:
        return {
            "word": word,
            "status": "insufficient_evidence",
            "appearance": appearance_month,
            "breakout": breakout_month,
            "breakout_info": breakout_info,
            "n_contexts": len(ctx),
            "old_count": old_count,
            "new_count": new_count,
            "drift_score": drift_score,
            "ctx": ctx,
            "emb": emb
        }

    # 6) clustering
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean"
    )
    labels = clusterer.fit_predict(emb)

    ctx_clustered = ctx.copy()
    ctx_clustered["cluster"] = labels
    ctx_clustered["period"] = ctx_clustered["month"].apply(assign_period)

    cluster_period_counts = pd.crosstab(ctx_clustered["cluster"], ctx_clustered["period"])
    cluster_period_props = pd.crosstab(
        ctx_clustered["cluster"],
        ctx_clustered["period"],
        normalize="columns"
    )

    return {
        "word": word,
        "status": "ok",
        "appearance": appearance_month,
        "breakout": breakout_month,
        "breakout_info": breakout_info,
        "n_contexts": len(ctx),
        "old_count": old_count,
        "new_count": new_count,
        "drift_score": drift_score,
        "ctx": ctx,
        "emb": emb,
        "clustered_df": ctx_clustered,
        "cluster_period_counts": cluster_period_counts,
        "cluster_period_props": cluster_period_props
    }

In [ ]:
def save_analysis_result(result):
    word = result["word"]

    if "ctx" in result and result["ctx"] is not None:
        result["ctx"].to_csv(
            os.path.join(CONTEXT_DIR, f"{word}_contexts.csv"),
            index=False
        )

    if "emb" in result and result["emb"] is not None:
        np.save(
            os.path.join(EMBED_DIR, f"{word}_embeddings.npy"),
            result["emb"]
        )

    if result.get("status") == "ok":
        result["clustered_df"].to_csv(
            os.path.join(CLUSTER_DIR, f"{word}_clusters.csv"),
            index=False
        )
        result["cluster_period_counts"].to_csv(
            os.path.join(CLUSTER_DIR, f"{word}_cluster_period_counts.csv")
        )
        result["cluster_period_props"].to_csv(
            os.path.join(CLUSTER_DIR, f"{word}_cluster_period_props.csv")
        )

In [ ]:
def analyze_word_live(
    word,
    df_counts,
    subreddit_dir,
    subreddit_name,
    per_month_limit=200,
    window_size=40,
    min_chars=40,
    old_end="2019-12",
    new_start="2023-01",
    min_cluster_size=20,
    min_samples=5
):

    word = word.lower().strip()

    # appearance
    appearance_month, _ = find_appearance_month(
        df_counts, word, min_count=20, sustain_months=2
    )

    # breakout
    breakout_month, breakout_info = find_breakout_month(
        df_counts, word, appearance_month,
        min_count=30, growth_ratio=3.0, lookback=3
    )

    # contexts
    ctx = extract_local_contexts_balanced(
        word=word,
        subreddit_dir=subreddit_dir,
        subreddit_name=subreddit_name,
        per_month_limit=per_month_limit,
        window_size=window_size,
        min_chars=min_chars
    )

    if ctx.empty:
        return {
            "word": word,
            "status": "no_contexts_found",
            "appearance": appearance_month,
            "breakout": breakout_month
        }

    # embeddings
    texts = ctx["text"].tolist()

    emb = embed_model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    # drift
    drift_score, old_count, new_count = compute_drift_score(
        ctx,
        emb,
        old_end=old_end,
        new_start=new_start
    )

    # clustering
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean"
    )

    labels = clusterer.fit_predict(emb)

    ctx_clustered = ctx.copy()
    ctx_clustered["cluster"] = labels
    ctx_clustered["period"] = ctx_clustered["month"].apply(assign_period)

    cluster_period_counts = pd.crosstab(
        ctx_clustered["cluster"],
        ctx_clustered["period"]
    )

    cluster_period_props = pd.crosstab(
        ctx_clustered["cluster"],
        ctx_clustered["period"],
        normalize="columns"
    )

    return {
        "word": word,
        "status": "ok",
        "appearance": appearance_month,
        "breakout": breakout_month,
        "breakout_info": breakout_info,
        "drift_score": drift_score,
        "old_count": old_count,
        "new_count": new_count,
        "ctx": ctx,
        "emb": emb,
        "clustered_df": ctx_clustered,
        "cluster_period_counts": cluster_period_counts,
        "cluster_period_props": cluster_period_props
    }

In [ ]:
def analyze_word(word):
    word = word.lower().strip()

    context_path = os.path.join(CONTEXT_DIR, f"{word}_contexts.csv")
    embed_path = os.path.join(EMBED_DIR, f"{word}_embeddings.npy")
    cluster_path = os.path.join(CLUSTER_DIR, f"{word}_clusters.csv")
    cluster_counts_path = os.path.join(CLUSTER_DIR, f"{word}_cluster_period_counts.csv")
    cluster_props_path = os.path.join(CLUSTER_DIR, f"{word}_cluster_period_props.csv")

    already_processed = (
        os.path.exists(context_path) and
        os.path.exists(embed_path)
    )

    if already_processed:
        print(f"Loading saved analysis for '{word}'")
        return query_word(word)

    print(f"No saved analysis found for '{word}'. Running live analysis...")

    result = analyze_word_live(
        word=word,
        df_counts=df,
        subreddit_dir=subreddit_dir,
        subreddit_name=SUBREDDIT,
        per_month_limit=200,
        window_size=40,
        min_chars=40
    )

    if result.get("status") in ["ok", "insufficient_evidence"]:
        save_analysis_result(result)

    return result

In [ ]:
def print_analysis_result(result):
    if result is None:
        print("No result")
        return

    print(f"Word: {result['word']}")
    print(f"Status: {result.get('status', 'loaded')}")
    print(f"Appearance: {result.get('appearance')}")
    print(f"Breakout: {result.get('breakout')}")
    print(f"Breakout info: {result.get('breakout_info')}")
    print(f"Drift score: {result.get('drift_score')}")
    print(f"Contexts: {result.get('n_contexts', len(result['ctx']) if 'ctx' in result else None)}")
    print(f"Old contexts: {result.get('old_count')}")
    print(f"New contexts: {result.get('new_count')}")

In [ ]:
result = analyze_word("diffusion")
print_analysis_result(result)

In [ ]:
save_analysis_result(result)
print("Saved live analysis for diffusion")

In [ ]:
def query_word(word):
    word = word.lower().strip()

    context_path = os.path.join(CONTEXT_DIR, f"{word}_contexts.csv")
    embed_path = os.path.join(EMBED_DIR, f"{word}_embeddings.npy")
    cluster_path = os.path.join(CLUSTER_DIR, f"{word}_clusters.csv")
    cluster_counts_path = os.path.join(CLUSTER_DIR, f"{word}_cluster_period_counts.csv")
    cluster_props_path = os.path.join(CLUSTER_DIR, f"{word}_cluster_period_props.csv")

    if not os.path.exists(context_path):
        print(f"No precomputed contexts found for '{word}'")
        return None

    ctx = pd.read_csv(context_path)

    emb = None
    if os.path.exists(embed_path):
        emb = np.load(embed_path)

    clustered_df = None
    if os.path.exists(cluster_path):
        clustered_df = pd.read_csv(cluster_path)

    cluster_period_counts = None
    if os.path.exists(cluster_counts_path):
        cluster_period_counts = pd.read_csv(cluster_counts_path, index_col=0)

    cluster_period_props = None
    if os.path.exists(cluster_props_path):
        cluster_period_props = pd.read_csv(cluster_props_path, index_col=0)

    appearance_month, _ = find_appearance_month(
        df, word, min_count=20, sustain_months=2
    )

    breakout_month, breakout_info = find_breakout_month(
        df, word, appearance_month, min_count=30, growth_ratio=3.0, lookback=3
    )

    drift_score = None
    old_count, new_count = 0, 0
    if emb is not None:
        drift_score, old_count, new_count = compute_drift_score(
            ctx, emb, old_end=OLD_END, new_start=NEW_START
        )

    return {
        "word": word,
        "status": "loaded",
        "appearance": appearance_month,
        "breakout": breakout_month,
        "breakout_info": breakout_info,
        "drift_score": drift_score,
        "old_count": old_count,
        "new_count": new_count,
        "ctx": ctx,
        "emb": emb,
        "clustered_df": clustered_df,
        "cluster_period_counts": cluster_period_counts,
        "cluster_period_props": cluster_period_props
    }

In [ ]:
def show_result_samples(result, n=5):
    if result is None or "ctx" not in result or result["ctx"] is None:
        print("No contexts available")
        return

    ctx = result["ctx"]
    old_mask = ctx["month"] <= OLD_END
    new_mask = ctx["month"] >= NEW_START

    print("\nOLD SAMPLES")
    if int(old_mask.sum()) > 0:
        display(ctx[old_mask][["month", "text"]].sample(min(n, int(old_mask.sum())), random_state=42))
    else:
        print("No old samples")

    print("\nNEW SAMPLES")
    if int(new_mask.sum()) > 0:
        display(ctx[new_mask][["month", "text"]].sample(min(n, int(new_mask.sum())), random_state=42))
    else:
        print("No new samples")

In [ ]:
def show_result_clusters(result, n=5):
    if result is None or result.get("clustered_df") is None:
        print("No usable cluster result")
        return

    clustered_df = result["clustered_df"]

    print("Cluster counts:")
    print(clustered_df["cluster"].value_counts().sort_index())

    if result.get("cluster_period_counts") is not None:
        print("\nCluster by period (raw counts):")
        display(result["cluster_period_counts"])

    if result.get("cluster_period_props") is not None:
        print("\nCluster by period (proportions):")
        display(result["cluster_period_props"])

    for c in sorted(clustered_df["cluster"].unique()):
        if c == -1:
            continue
        print(f"\n===== CLUSTER {c} =====")
        display(
            clustered_df[clustered_df["cluster"] == c][["month", "text"]]
            .sample(min(n, (clustered_df["cluster"] == c).sum()), random_state=42)
        )

In [ ]:
def print_analysis_result(result):
    if result is None:
        print("No result")
        return

    print(f"Word: {result['word']}")
    print(f"Status: {result.get('status')}")
    print(f"Appearance: {result.get('appearance')}")
    print(f"Breakout: {result.get('breakout')}")
    print(f"Breakout info: {result.get('breakout_info')}")
    print(f"Drift score: {result.get('drift_score')}")
    print(f"Old contexts: {result.get('old_count')}")
    print(f"New contexts: {result.get('new_count')}")

In [ ]:
result = analyze_word("diffusion")

In [ ]:
show_result_samples(result, n=5)
show_result_clusters(result, n=5)

In [ ]:
result = analyze_word("transformer")
print_analysis_result(result)

In [ ]:
result = analyze_word("transformer")

In [ ]:
show_result_samples(result, n=5)
show_result_clusters(result, n=5)

In [ ]:
result = analyze_word("classification")
print_analysis_result(result)